# Synapse & Axon Demo

This notebook demonstrates how **Axon** (a high-level DSL), **Synapse** (a YAML graph spec), and the **PyTorch / TinyGrad backends** work together in brainsurgery.

**Flow:** Axon `.axon` file → Synapse `.yaml` spec → Backend (PyTorch or TinyGrad) → Runnable Python code

## 1. Setup

In [4]:
from pathlib import Path
from omegaconf import OmegaConf

from brainsurgery.synapse import emit_model_code_from_synapse_spec
from brainsurgery.synapse.backends import list_backends

EXAMPLES = Path("../examples") if Path("../examples").exists() else Path("examples")
print("Available backends:", list_backends())

Available backends: ['pytorch', 'tinygrad']


## 2. The Axon Source

Axon is a compact DSL for defining model architectures. Here's GPT-2:

In [5]:
axon_path = EXAMPLES / "gpt2.axon"
print(axon_path.read_text())

T   = 1024   -- max sequence length
D   = 768    -- hidden dimensions
L   = 12     -- number of layers
H   = 12     -- number of attention heads
V   = 520257 -- vocab size

import Activations gelu_new

lin :: @Path -> Tensor[B,S,Din] -> I -> Tensor[B,S,dim]
lin@path x dim = linear@path x dim=dim bias=true transpose=true

gpt2_block :: Tensor[B,S,D] -> ?Tensor[B,S] -> Tensor[B,S,D]
gpt2_block x attn_mask = do
  x1 <- layernorm@ln_1 x
  a <- scope@attn do
    q_lin, k_lin, v_lin <- lin@c_attn x1 3*D |> split parts=3
    q <- reshape_heads q_lin heads=H
    k <- reshape_heads k_lin heads=H
    v <- reshape_heads v_lin heads=H
    mask <- causal_mask q k window=768 padding_mask=attn_mask
    return attention q k v mask=mask |> merge_heads |> lin@c_proj D
  x <- x + a
  x3 <- layernorm@ln_2 x
  m <- scope@mlp do
    return lin@c_fc x3 4*D |> gelu_new fp32_accum=true |> lin@c_proj D
  return x + m

gpt2 :: TokenIds[B,S] -> ?Tensor[B,S] -> Tensor[B,S,V]
gpt2 input_ids attn_mask = do
  tok <- 

Key features:
- **Symbols** (`T`, `D`, `L`, `H`, `V`) define dimensions at the top
- **`lin`** is a helper function wrapping `linear` with common defaults
- **`gpt2_block`** defines a transformer block (attention + MLP) with residual connections
- **`gpt2`** is the top-level module composing embedding → blocks → norm → project
- `@path` annotations map to weight names in a state dict
- `?Tensor` marks optional inputs (like attention masks)

## 3. Lowering Axon → Synapse

The Axon source is lowered into a Synapse YAML spec — a machine-readable computational graph.

In [6]:
from brainsurgery.synapse import parse_axon_program_from_path, lower_axon_program_to_synapse_spec

parsed = parse_axon_program_from_path(axon_path)
spec = lower_axon_program_to_synapse_spec(parsed)

# Show the top-level structure (symbols, inputs, outputs)
model = spec["model"]
print("Symbols:", model.get("symbols"))
print("Inputs:", list(model.get("inputs", {}).keys()))
print("Outputs:", model.get("outputs"))
print(f"Graph nodes: {len(model['graph'])}")
print(f"Blocks: {list(model.get('blocks', {}).keys())}")

Symbols: {'T': 1024, 'D': 768, 'L': 12, 'H': 12, 'V': None, 'B': None, 'S': None}
Inputs: ['input_ids', 'attn_mask']
Outputs: {'logits': 'logits'}
Graph nodes: 7
Blocks: ['gpt2_block']


The lowered Synapse YAML has concrete values for all symbols and an explicit graph of op nodes. You can also load a pre-lowered spec:

In [7]:
synapse_path = EXAMPLES / "gpt2_synapse.yaml"
spec_from_yaml = OmegaConf.to_container(OmegaConf.load(synapse_path), resolve=True)
print(f"Loaded synapse spec with {len(spec_from_yaml['model']['graph'])} top-level nodes")

Loaded synapse spec with 7 top-level nodes


## 4. Emitting PyTorch Code

Pass the spec to the PyTorch backend to generate an `nn.Module` class.

In [8]:
pytorch_src = emit_model_code_from_synapse_spec(spec, class_name="GPT2", backend="pytorch")

# Show key parts of the generated code
lines = pytorch_src.split("\n")
for line in lines[:25]:
    print(line)
print("  ...")
print(f"\nTotal lines: {len(lines)}")

from __future__ import annotations

from typing import Any

from brainsurgery.synapse.mxfp4 import materialize_mxfp4_aliases

import math
import torch
from torch import nn
from torch.nn import functional as F


class GPT2(nn.Module):
    def __init__(self, state_dict: dict[str, torch.Tensor] | None = None) -> None:
        super().__init__()
        self._state: dict[str, torch.Tensor] = {}
        self._symbols: dict[str, int | float | bool] = {'T': 1024, 'D': 768, 'L': 12, 'H': 12}
        self._trace_enabled = False
        self.trace_ops: list[dict[str, Any]] = []
        if state_dict is not None:
            self.load_state_dict_tensors(state_dict)

    @classmethod
    def from_state_dict(cls, state_dict: dict[str, torch.Tensor]) -> "GPT2":
        return cls(state_dict=state_dict)
  ...

Total lines: 509


In [9]:
# The generated class is an nn.Module with a generate() method
assert "class GPT2(nn.Module):" in pytorch_src
assert "def forward(" in pytorch_src
assert "def generate(" in pytorch_src
print("PyTorch backend produces nn.Module with forward() and generate()")

PyTorch backend produces nn.Module with forward() and generate()


## 5. Emitting TinyGrad Code

The same spec can be compiled for TinyGrad — a completely different runtime.

In [10]:
tinygrad_src = emit_model_code_from_synapse_spec(spec, class_name="GPT2Tiny", backend="tinygrad")

lines = tinygrad_src.split("\n")
for line in lines[:25]:
    print(line)
print("  ...")
print(f"\nTotal lines: {len(lines)}")

NotImplementedError: Unsupported op in TinyGrad codegen: 'activations_gelu_new'

In [ ]:
# TinyGrad produces a plain class (no nn.Module), using tinygrad.Tensor
assert "class GPT2Tiny:" in tinygrad_src
assert "from tinygrad import Tensor" in tinygrad_src
assert "nn.Module" not in tinygrad_src
print("TinyGrad backend produces a plain class with tinygrad.Tensor ops")

## 6. Comparing the Two Backends

Both backends consume the same Synapse spec but produce different code:

In [ ]:
print(f"{'':>15} {'PyTorch':>12} {'TinyGrad':>12}")
print(f"{'Lines':>15} {len(pytorch_src.splitlines()):>12} {len(tinygrad_src.splitlines()):>12}")
print(f"{'Has nn.Module':>15} {'yes':>12} {'no':>12}")
print(f"{'Has generate()':>15} {'yes':>12} {'yes':>12}")
print(f"{'Tensor type':>15} {'torch.Tensor':>12} {'tinygrad.Tensor':>12}")
print(f"{'State loading':>15} {'load_state_dict':>12} {'state_dict dict':>12}")

## 7. Round-Trip: Synapse → Axon

You can also render a Synapse YAML spec back to readable Axon source.

In [ ]:
from brainsurgery.synapse import synapse_spec_to_axon_module_text

axon_text = synapse_spec_to_axon_module_text(spec, module_name="gpt2")
print(axon_text)

## 8. Real-World Spec: Gemma 3 270M

The same pipeline works for production architectures like Gemma 3, which includes KV-caching, RoPE, sliding window attention, and GQA.

In [ ]:
gemma_path = EXAMPLES / "gemma3_270m_synapse.yaml"
gemma_spec = OmegaConf.to_container(OmegaConf.load(gemma_path), resolve=True)
gmodel = gemma_spec["model"]

print("Gemma 3 270M symbols:")
for k, v in gmodel["symbols"].items():
    print(f"  {k}: {v}")
print(f"\nTop-level graph nodes: {len(gmodel['graph'])}")
print(f"Blocks: {list(gmodel.get('blocks', {}).keys())}")

In [ ]:
# Compare PyTorch vs TinyGrad output size for Gemma 3
gemma_pt = emit_model_code_from_synapse_spec(gemma_spec, class_name="Gemma3", backend="pytorch")
gemma_tg = emit_model_code_from_synapse_spec(gemma_spec, class_name="Gemma3Tiny", backend="tinygrad")

print(f"Gemma 3 270M generated code:")
print(f"  PyTorch:  {len(gemma_pt.splitlines())} lines")
print(f"  TinyGrad: {len(gemma_tg.splitlines())} lines")

## 9. Summary

| Concept | Role | Format |
|---------|------|--------|
| **Axon** | Human-written model DSL | `.axon` text files |
| **Synapse** | Machine-readable graph spec | `.yaml` (versioned as `synapse: 1`) |
| **PyTorch backend** | Generates `nn.Module` code | `class Model(nn.Module):` |
| **TinyGrad backend** | Generates plain Tensor code | `class Model:` |

**The key insight:** one spec, multiple runtimes. Write your architecture once in Axon or Synapse YAML, then target PyTorch for training/inference or TinyGrad for lightweight execution — without changing the model definition.